# AnswerRelevancyMetric

## What it measures

Whether the answer actually addresses the question that was asked. The judge splits
`actual_output` into statements and asks, of each one, whether it is relevant to `input`.
The score is the proportion of relevant statements. It says nothing about whether the
answer is *true* - a fluent, on-topic fabrication scores 1.0 here. Pair it with
`FaithfulnessMetric` or `HallucinationMetric` for truth.

## When it is useful

As the first gate on any question-answering surface. It catches the failure where a
retrieval-grounded assistant pads its answer with policy boilerplate that is topically
adjacent but does not answer the analyst's question - a common regression when a prompt
or a retrieval filter changes.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` (single-turn) |
| `input` | yes |
| `actual_output` | yes |

No golden, no retrieval context, no tools. That makes this the cheapest metric in the
suite to run against a black-box API.

In [3]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

API base URL       : http://localhost:8000
Request timeout    : 180.0s
Judge model        : gpt-5.4-mini
Expected seed      : scenarios-v1
API key configured : False  (False is correct when AUTH_MODE=off)
OPENAI_API_KEY set : True


In [4]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
EXPECTED_SCHEMA_VERSION = "1.0.0"

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    if served and served != EXPECTED_SCHEMA_VERSION:
        print(f"WARNING: application reports contract version {served}, these "
              f"notebooks were written against {EXPECTED_SCHEMA_VERSION}. "
              f"Field names may have changed - see docs/evaluation-contract.md.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

----- GET /api/health -----
{
  "status": "ok",
  "schema_version": "1.0.0",
  "eval_tracing": false
}


## Endpoint exercised

`POST /api/rag/query` - the analyst-facing "ask a question about policy or evidence"
flow. It retrieves policy and case-scoped evidence chunks and generates a cited answer.

Two request choices matter for reproducibility:

- **`include_history: false`.** This endpoint is *stateful* when a `case_id` is given:
  every case has one conversation thread, and the string actually embedded for retrieval
  is the history-expanded query, not `question`. Sending `include_history: false` answers
  from this question alone, so the run does not depend on whatever was asked before.
  The turn is still recorded; it is simply not fed back in.
- **`case_id` resolved from `GET /api/eval/scenarios`**, never hardcoded.

In [3]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

s1: case_id=1 customer_id=1 transaction_id=1  Low-risk domestic payment
s2: case_id=2 customer_id=2 transaction_id=2  High-risk jurisdiction transfer
s3: case_id=3 customer_id=3 transaction_id=3  Possible sanctions name match (beneficiary near-miss)
s4: case_id=4 customer_id=4 transaction_id=4  Incomplete source-of-funds evidence
s5: case_id=5 customer_id=5 transaction_id=5  PEP indicator
s6: case_id=6 customer_id=6 transaction_id=9  Structuring behaviour (four sub-threshold cash deposits)
s7: case_id=7 customer_id=7 transaction_id=10  Adverse media false positive
s8: case_id=8 customer_id=8 transaction_id=11  Contradictory invoice / payment purpose


In [4]:
# --------------------------------------------------------------------------
# The exact request. Secrets are masked by `show`; there are none in this body.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s6"]["case_id"]      # "Structuring behaviour" scenario

QUESTION = (
    "What indicators identify structuring, and what cash transaction thresholds "
    "require review under the transaction monitoring policy?"
)

request_body = {
    "question": QUESTION,
    "case_id": CASE_ID,
    "top_k": 8,
    "include_history": False,   # deterministic single-turn run
}

print("POST", f"{API_BASE}/api/rag/query")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body:", json.dumps(request_body, indent=2))

POST http://localhost:8000/api/rag/query
headers: {
  "X-API-Key": null,
  "Content-Type": "application/json"
}
body: {
  "question": "What indicators identify structuring, and what cash transaction thresholds require review under the transaction monitoring policy?",
  "case_id": 6,
  "top_k": 8,
  "include_history": false
}


In [5]:
# --------------------------------------------------------------------------
# The raw response.
# --------------------------------------------------------------------------
response = api("POST", "/api/rag/query", json_body=request_body)

show("POST /api/rag/query", {k: v for k, v in response.items()
                             if k != "retrieved_context"})
print()
print(f"retrieved_context: {len(response['retrieved_context'])} chunks - "
      f"{[c['chunk_id'] for c in response['retrieved_context']]}")

----- POST /api/rag/query -----
{
  "run_id": 9,
  "kind": "rag_query",
  "created_at": "2026-07-29T11:06:02.687240",
  "question": "What indicators identify structuring, and what cash transaction thresholds require review under the transaction monitoring policy?",
  "retrieval_query": "What indicators identify structuring, and what cash transaction thresholds require review under the transaction monitoring policy?",
  "answer": "The indicators that identify structuring include multiple cash deposits made over consecutive days that are just below the GBP 10,000 single transaction threshold, as seen in the cash deposit log where all four deposits are within 10% below this threshold. The cash transaction threshold requiring review under the transaction monitoring policy is GBP 10,000, as indicated by the AML-001 s2.3 indicator.",
  "citations": [
    {
      "chunk_number": 2,
      "chunk_id": "doc14-c0",
      "document_id": 14,
      "source": "Branch operations export"
    }
  ],
  "

## Mapping the API response onto DeepEval fields

| DeepEval field | API field | Note |
|---|---|---|
| `input` | `question` | Echoed by the API, so what is scored is what the application received |
| `actual_output` | `answer` | The generated (or deterministic refusal) answer text |

Fields deliberately **not** used here: `retrieved_context` (that is Faithfulness and
Contextual Precision/Recall territory), `citations`, and `grounding`. Answer Relevancy is
purely a question-answer alignment check.

One case is worth calling out. When retrieval returns nothing the application returns a
fixed refusal string with `grounding: "insufficient_evidence"`, `model: "none"` and
`usage: null` - the LLM is verifiably never invoked. That refusal is the *correct*
behaviour, but Answer Relevancy will score it near zero because a refusal contains no
statements relevant to the question. The cell below asserts we are not on that path, so a
low score can only mean a genuine relevance problem.

In [6]:
# --------------------------------------------------------------------------
# Guard: make sure we are scoring a generated answer, not the deterministic
# refusal. Scoring the refusal would produce a meaningless failure.
# --------------------------------------------------------------------------
print("grounding :", response["grounding"])
print("model     :", response["model"])
print("citations :", [c["chunk_id"] for c in response["citations"]])

if response["grounding"] == "insufficient_evidence":
    raise RuntimeError(
        "Retrieval returned no chunks, so the application returned its deterministic "
        "refusal and never called the LLM. AnswerRelevancyMetric cannot say anything "
        "useful about a refusal.\n"
        "Check that the vector index is built (POST /api/dev/reset) and that the "
        "question matches indexed content."
    )

grounding : grounded
model     : openai/gpt-4o-mini
citations : ['doc14-c0']


## No golden is required

Answer Relevancy is reference-free. The judge compares the answer against the question
alone, so there is nothing to derive and nothing that could be tuned to make the metric
pass. That is the whole reason it is a good first gate.

In [7]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=response["question"],
    actual_output=response["answer"],
)

print("USER INPUT")
print(textwrap.fill(test_case.input, width=96, initial_indent="  ", subsequent_indent="  "))
print()
print("ACTUAL OUTPUT")
print(textwrap.fill(test_case.actual_output, width=96, initial_indent="  ",
                    subsequent_indent="  "))
print()
print("EXPECTED OUTPUT (golden) : not used by this metric")
print("RETRIEVAL CONTEXT        : not used by this metric")

USER INPUT
  What indicators identify structuring, and what cash transaction thresholds require review
  under the transaction monitoring policy?

ACTUAL OUTPUT
  The indicators that identify structuring include multiple cash deposits made over consecutive
  days that are just below the GBP 10,000 single transaction threshold, as seen in the cash
  deposit log where all four deposits are within 10% below this threshold. The cash transaction
  threshold requiring review under the transaction monitoring policy is GBP 10,000, as indicated
  by the AML-001 s2.3 indicator.

EXPECTED OUTPUT (golden) : not used by this metric
RETRIEVAL CONTEXT        : not used by this metric


## Judge and threshold

- **Judge model**: taken from `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `AnswerRelevancyMetric`, passed
  explicitly so the notebook states its own pass bar rather than inheriting it silently.

The default is kept deliberately. Answer Relevancy is a proportion of relevant
statements, so `0.5` means "more than half of what the assistant said addressed the
question" - a floor, not a quality target. Raising it here would turn a smoke test into a
style test, and style belongs in a `GEval` metric with explicit criteria.

In [8]:
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,       # deterministic cell ordering under nbclient
    verbose_mode=True,      # populates metric.verbose_logs for debugging
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

metric class : AnswerRelevancyMetric
judge model  : gpt-5.4-mini
threshold    : 0.5
async_mode   : False
strict_mode  : False


In [9]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

/Users/bola/seyi/AI-LLM/aml-kyc-agentic-platform/.venv/lib/python3.14/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

**************************************************

Answer Relevancy Verbose Logs

**************************************************

Statements:
[
    "Indicators that identify structuring include multiple cash deposits made over consecutive days just below the 
GBP 10,000 single transaction threshold.",
    "The cash deposit log shows all four deposits are within 10% below the threshold.",
    "The cash transaction threshold requiring review under the transaction monitoring policy is GBP 10,000.",
    "AML-001 s2.3 indicates this threshold."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "This is supporting detail about the deposit pattern and threshold proximity, but it does not 
directly answer the question beyond reinforcing the structuring indicator."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "This cites a policy section as support, but the section reference itself is not the direct 
answer to the question."
    }
]
 
Score: 1.0
Reason: The score is 1.00 because the response is fully relevant to the question, with no irrelevant statements 
included. It directly addresses the requested indicators and transaction thresholds, so there’s nothing to deduct.

======================================================================

In [10]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

metric          : AnswerRelevancyMetric
judge model     : gpt-5.4-mini
threshold       : 0.5
score           : 1.0
PASS / FAIL     : PASS
judge cost (USD): 0.0019672500000000002

reason:
The score is 1.00 because the response is fully relevant to the question, with no irrelevant
  statements included. It directly addresses the requested indicators and transaction
  thresholds, so there’s nothing to deduct.

----- verbose judge log (debug) -----
Statements:
[
    "Indicators that identify structuring include multiple cash deposits made over consecutive days just below the GBP 10,000 single transaction threshold.",
    "The cash deposit log shows all four deposits are within 10% below the threshold.",
    "The cash transaction threshold requiring review under the transaction monitoring policy is GBP 10,000.",
    "AML-001 s2.3 indicates this threshold."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "This i

## Limitations in a black-box acceptance test

1. **Relevance is not correctness.** A confident, on-topic answer built from nothing
   scores 1.0. This metric must never be the only gate on an AI endpoint.
2. **The judge is itself an LLM.** Scores vary run to run. Treat a single score as a
   smoke signal and a sustained drop across runs as a regression; do not chase decimals.
3. **Retrieval is a hidden confound.** A low score may mean the generator rambled, or it
   may mean retrieval returned nothing usable and the generator hedged. `POST
   /api/rag/retrieve` isolates the retrieval half with no LLM involved - use it before
   concluding the generator is at fault.
4. **The refusal path scores badly by construction.** The guard cell above turns that
   into an explicit error rather than a silent low score, but any suite that sweeps many
   questions must special-case `grounding == "insufficient_evidence"` or it will report
   correct refusals as failures.
5. **Conversation memory can leak in.** Only `include_history: false` makes the run
   independent of earlier turns on the same case. A suite that omits it is scoring a
   different, history-expanded query than the one it printed.